3.1 Import of the Library

In [ ]:
!pip install pandas numpy statsmodels matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False

3.2 Data Input

In [ ]:
file_path = r"C:\Users\cgss2023.dta"
df_raw = pd.read_stata(file_path, convert_categoricals=False)

3.3 Variable Selection

In [ ]:
vars_map = {
    'a36': 'happiness',      # Dependent variable
    'a8a': 'income',         # Independent variable
    'a15': 'health',
    'a7a': 'education',
    'a69': 'marital_status',
    'a43a': 'social_class',
    'a25a': 'residence',
    'a285': 'internet_freq',
    'a2': 'gender',
    'a10': 'politics',
    'a3a': 'birth_year',
}
df = df_raw[vars_map.keys()].rename(columns=vars_map).copy()

3.4 Data Cleaning

In [ ]:
#3.4.1 Mark all values less than 0 as missing
for col in df.columns:
    # Only apply the less-than-zero judgment to numeric columns to avoid errors when comparing strings with integers
    if pd.api.types.is_numeric_dtype(df[col]):
        df.loc[df[col] < 0, col] = pd.NA
#3.4.2 Special rule processing
df.loc[df['happiness'] > 5, 'happiness'] = pd.NA
df.loc[df['income'] >= 9999996, 'income'] = pd.NA
df.loc[df['education'] == 14, 'education'] = pd.NA
df['age'] = 2023 - df['birth_year']
#3.4.3 Variable recoding
# Education level
edu_map = {
    1: "Primary school or below", 2: "Primary school or below", 3: "Primary school or below", 4: "Junior high school",
    5: "High school_Vocational school", 6: "High school_Vocational school", 7: "High school_Vocational school", 8: "High school_Vocational school",
    9: "Junior college", 10: "Junior college", 11: "Undergraduate", 12: "Undergraduate", 13: "Graduate and above"
}
marry_map = {1: 'Unmarried', 2: 'Unmarried', 3: 'Married', 4: 'Married', 5: 'Married', 6: 'Divorced', 7: 'Widowed'}
res_map = {1: 'Urban_City and town', 2: 'Urban_City and town', 3: 'Township', 4: 'Township'}
pol_map = {1: 'Non-party member', 2: 'Non-party member', 3: 'Non-party member', 4: 'Communist Party member'}

df['education'] = df['education'].map(edu_map)
df['marital_status'] = df['marital_status'].map(marry_map)
df['residence'] = df['residence'].map(res_map)
df['politics'] = df['politics'].map(pol_map)

# Take logarithm of income
df['income'] = np.log1p(df['income'].astype(float))
#3.4.4 Delete all rows containing missing values
df.dropna(inplace=True)


In [ ]:
print(df.head(10).to_string()) # to_string确保打印对齐